In [1]:
import pandas as pd
import requests
from openai import OpenAI
import json
import time
from apikeys import open_ai_api_key

# Initialize OpenAI client
client = OpenAI(api_key=open_ai_api_key)

# Read the project summary CSV
df = pd.read_csv('project_summary.csv')

# Filter where Other_Documents_Count != 0
filtered_df = df[df['Other_Documents_Count'] != 0]

print(f"Total projects with other documents: {len(filtered_df)}")
print(f"First 5 projects:")
print(filtered_df[['Project_Name', 'First_File_URL', 'Other_Documents_Count']].head())

/Users/ghinwamoujaes/Desktop/World Bank/Code/slovakia_CaseStudy/wb_venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Total projects with other documents: 67
First 5 projects:
                                         Project_Name  \
1   Aktualizácia hodnotenia projektu Bojových obrn...   
2   Akvizícia palebných prostriedkov protivzdušnej...   
3   Akvizícia prostriedkov protivzdušnej obrany (p...   
5                          Budova Najvyššieho súdu SR   
15                 D1 Beharovce – Branisko, 2. profil   

                                       First_File_URL  Other_Documents_Count  
1   https://www.mfsr.sk/files/archiv/68/Aktualizac...                      1  
2   https://www.mfsr.sk/files/archiv/6/UHP_MO_Aktu...                      1  
3   https://www.mfsr.sk/files/archiv/93/Hodnotenie...                      1  
5   https://www.mfsr.sk/files/archiv/78/Hodnotenie...                      1  
15  https://www.mfsr.sk/files/sk/financie/hodnota-...                      1  


In [4]:
import PyPDF2
import os

def extract_text_from_pdf(pdf_path):
    """
    Extract text from a PDF file
    """
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            text = ""
            for page in pdf_reader.pages:
                text += page.extract_text() + "\n"
            return text
    except Exception as e:
        return f"Error reading PDF: {str(e)}"

def analyze_document_with_openai(document_id, project_name):
    """
    Analyze a document using OpenAI API by reading the local PDF file
    """
    try:
        # Construct the path to the PDF file
        pdf_path = f"downloaded_first_links/{document_id}.pdf"
        
        # Check if file exists
        if not os.path.exists(pdf_path):
            return f"PDF file not found: {pdf_path}"
        
        # Extract text from PDF
        pdf_text = extract_text_from_pdf(pdf_path)
        
        if pdf_text.startswith("Error reading PDF"):
            return pdf_text
        
        # Limit text length to avoid token limits (keep first 8000 characters)
        if len(pdf_text) > 8000:
            pdf_text = pdf_text[:8000] + "... [text truncated]"
        
        prompt = f"""This document is an assessment of a Cost Benefit Analysis project in Slovak language. 
        Read through the document and extract the following information - Extract only in JSON file. 
        Make sure that the responses come from the document and do not make anything up.

        Project Name: {project_name}
        Document ID: {document_id}

        Document Text:
        {pdf_text}

        Please extract:
        - Positive/Negative/Neutral Recommendation of the Project - Provide some additional information about the recommendation to support the answer
        - Is the Feasibility Study Considered Complete?
        - EIIR estimated or CBA Estimate Result - Estimated EIIR internal rate of return value based on the document/or Benefit-Cost Ratio and project being analyzed
        - EIIR above deemed viable - Provide some additional information about the recommendation to support the answer
        - Climate Change Risk Mentioned - Yes/No on whether the assessment mentions climate change risk
        - Climate Change Risk Mentioned Descriptions - Provide some additional information about the climate change risk to support the answer

        All responses should be in English. Return the response in JSON format with these exact keys:
        {{
            "recommendation": "Positive/Negative/Neutral"
            "recommendation_description": "Additional information about the recommendation to support the answer",
            "feasibility_study_complete": "Yes/No/Not specified",
            "eiir_estimated": "value or Not specified",
            "eiir_above_viable": "Yes/No/Not specified - provide some additional information about the recommendation to support the answer",
            "climate_change_risk_mentioned": "Yes/No",
            "climate_change_risk_mentioned_descriptions": "Additional information about the climate change risk to support the answer"
        }}"""
        
        # Call OpenAI API
        response = client.chat.completions.create(
            model="gpt-5",
            messages=[
                {"role": "system", "content": "You are an expert at analyzing Slovak government documents and extracting structured information. Always respond in JSON format."},
                {"role": "user", "content": prompt}
            ]
        )
        
        return response.choices[0].message.content
        
    except Exception as e:
        return f"Error analyzing document: {str(e)}"

# Get the first 5 documents
documents_to_analyze = filtered_df[30:]

print("\\nAnalyzing first 30 documents...")
print("=" * 50)

results = []

for index, row in documents_to_analyze.iterrows():
    project_name = row['Project_Name']
    document_id = row['First_File_Document_ID']
    url = row['First_File_URL']
    
    print(f"\\nAnalyzing: {project_name}")
    print(f"Document ID: {document_id}")
    print(f"URL: {url}")
    
    # Analyze the document using local PDF
    result = analyze_document_with_openai(document_id, project_name)
    
    # Store results
    results.append({
        'project_name': project_name,
        'document_id': document_id,
        'url': url,
        'analysis': result
    })
    
    print(f"Result: {result}")
    print("-" * 30)
    
    # Add a small delay to avoid rate limiting
    time.sleep(2)


\nAnalyzing first 30 documents...
\nAnalyzing: Obmena rádiolokačnej techniky VzS OS SR (stredný, malý a blízky dosah)
Document ID: 4.003
URL: https://www.mfsr.sk/files/archiv/15/Aktualizaciahodnotenia3DRadary_20210324.pdf
Result: {
  "recommendation": "Neutral",
  "recommendation_description": "The update states total costs remain 148.2 million EUR incl. VAT; additional support equipment (generators, transformers, remote control) worth 4.89 million EUR will be delivered, financed by a discount on radar prices and cancelling the simulator. Training will occur on the radar’s built-in simulator function. MF SR recommended checking the option to procure two radars for air defense; the MoD deemed it disadvantageous, and due to time constraints UHP could not verify the process. No explicit approval or rejection is stated.",
  "feasibility_study_complete": "Not specified",
  "eiir_estimated": "Not specified",
  "eiir_above_viable": "Not specified - The document provides no EIIR or benefit-cos

In [5]:
# Display results in a more organized format
print("\\n" + "="*80)
print("SUMMARY OF ANALYSIS RESULTS")
print("="*80)

# Prepare data for CSV
csv_data = []

for i, result in enumerate(results, 1):
    print(f"\\n{i}. PROJECT: {result['project_name']}")
    print(f"   Document ID: {result['document_id']}")
    print(f"   URL: {result['url']}")
    print(f"   ANALYSIS:")
    
    # Initialize row data
    row_data = {
        'Project_Name': result['project_name'],
        'Document_ID': result['document_id'],
        'URL': result['url'],
        'Recommendation': 'Not specified',
        'Recommendation_Description': 'Not specified',
        'Feasibility_Study_Complete': 'Not specified',
        'EIIR_Estimated': 'Not specified',
        'EIIR_Above_Viable': 'Not specified',
        'Climate_Change_Risk_Mentioned': 'Not specified',
        'Climate_Change_Risk_Descriptions': 'Not specified'
    }
    
    try:
        # Try to parse as JSON for better formatting
        analysis_json = json.loads(result['analysis'])
        for key, value in analysis_json.items():
            formatted_key = key.replace('_', ' ').title()
            print(f"     {formatted_key}: {value}")
            
            # Map to CSV columns
            if key == 'recommendation':
                row_data['Recommendation'] = value
            elif key == 'recommendation_description':
                row_data['Recommendation_Description'] = value
            elif key == 'feasibility_study_complete':
                row_data['Feasibility_Study_Complete'] = value
            elif key == 'eiir_estimated':
                row_data['EIIR_Estimated'] = value
            elif key == 'eiir_above_viable':
                row_data['EIIR_Above_Viable'] = value
            elif key == 'climate_change_risk_mentioned':
                row_data['Climate_Change_Risk_Mentioned'] = value
            elif key == 'climate_change_risk_mentioned_descriptions':
                row_data['Climate_Change_Risk_Descriptions'] = value
                
    except json.JSONDecodeError:
        # If not JSON, print as is
        print(f"     {result['analysis']}")
        row_data['Recommendation'] = result['analysis']
    
    csv_data.append(row_data)
    print("-" * 60)

# Convert to DataFrame and save as CSV
results_df = pd.DataFrame(csv_data)
results_df.to_csv('document_analysis_results_v4.csv', index=False, encoding='utf-8')

print(f"\\nResults saved to 'document_analysis_results.csv'")
print(f"Total documents analyzed: {len(results)}")

# Display the CSV structure
print("\\nCSV Structure:")
print(results_df.head())

\n================================================================================
SUMMARY OF ANALYSIS RESULTS
\n1. PROJECT: Obmena rádiolokačnej techniky VzS OS SR (stredný, malý a blízky dosah)
   Document ID: 4.003
   URL: https://www.mfsr.sk/files/archiv/15/Aktualizaciahodnotenia3DRadary_20210324.pdf
   ANALYSIS:
     Recommendation: Neutral
     Recommendation Description: The update states total costs remain 148.2 million EUR incl. VAT; additional support equipment (generators, transformers, remote control) worth 4.89 million EUR will be delivered, financed by a discount on radar prices and cancelling the simulator. Training will occur on the radar’s built-in simulator function. MF SR recommended checking the option to procure two radars for air defense; the MoD deemed it disadvantageous, and due to time constraints UHP could not verify the process. No explicit approval or rejection is stated.
     Feasibility Study Complete: Not specified
     Eiir Estimated: Not specified
     

In [6]:
results_df.head()

,Project_Name,Document_ID,URL,Recommendation,Recommendation_Description,Feasibility_Study_Complete,EIIR_Estimated,EIIR_Above_Viable,Climate_Change_Risk_Mentioned,Climate_Change_Risk_Descriptions
0,Obmena rádiolokačnej techniky VzS OS SR (stred...,4.003,https://www.mfsr.sk/files/archiv/15/Aktualizac...,Neutral,The update states total costs remain 148.2 mil...,Not specified,Not specified,Not specified - The document provides no EIIR ...,No,The assessment does not mention climate change...
1,Obnova a rozvoj Kúpeľov Sliač,3.015,https://www.mfsr.sk/files/archiv/55/Hodnotenie...,Neutral,"Proceed with Phase A (254 beds, ~€60m) and rea...",No,Not specified,Not specified - The document provides NPVs (−€...,No,The assessment does not mention climate change...
2,Obstaranie Bojových obrnených vozidiel 8x8 (ak...,4.007,https://www.mfsr.sk/files/archiv/77/Hodnotenie...,Neutral,The assessment does not issue an unconditional...,No,Not specified,Not specified - The document provides no EIIR ...,No,Climate risks are not discussed; the assessmen...
3,Obstaranie Pásových bojových obrnených vozidie...,4.008,https://www.mfsr.sk/files/archiv/56/Hodnotenie...,Neutral,The assessment acknowledges the project aligns...,No,Not specified,Not specified - The document provides no inter...,No,Climate change risks are not discussed; the as...
4,Obstaranie batériových elektrických jednotiek ...,1.111,https://www.mfsr.sk/files/sk/financie/hodnota-...,Neutral,MF SR considers the project aligned with the s...,No,Not specified,Not specified - The assessment does not provid...,No,The document discusses reducing environmental ...


In [7]:
df = pd.read_excel('MSFR_Data_AllTypes.xlsx')
df.head()

,Sector,Project Name,Type,URL,Date,File Size,Document_ID,Recommendation,Description of Recommendation,Feasibility_Study_Complete,EIIR_Estimated,EIIR_Above_Viable,Climate Change Mentioned
0,Obrana,UNKNOWN_PROJECT,Dohody o hraničných priechodoch,https://www.mfsr.sk/sk/dane-cla-uctovnictvo/co...,NaN,NaN,4.023,NaN,NaN,NaN,NaN,NaN,NaN
1,Obrana,Rekonštrukcia leteckej základne Sliač (aktuali...,hodnotenie,https://www.mfsr.sk/files/archiv/28/mo_sliac_a...,23.1.2025,140 kB,4.016,NaN,NaN,NaN,NaN,NaN,NaN
2,Obrana,Rekonštrukcia leteckej základne Sliač (aktuali...,štúdia uskutočniteľnosti,https://www.mosr.sk/data/files/5761_priloha-7-...,NaN,NaN,4.025,NaN,NaN,NaN,NaN,NaN,NaN
3,Obrana,Rekonštrukcia leteckej základne Sliač (aktuali...,hodnotenie,https://www.mfsr.sk/files/archiv/28/MO_Sliac_a...,4.10.2024,165 kB,4.015,NaN,NaN,NaN,NaN,NaN,NaN
4,Obrana,Rekonštrukcia leteckej základne Sliač (aktuali...,štúdia uskutočniteľnosti,https://www.mosr.sk/data/files/5691_semai-195-...,NaN,NaN,4.027,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
len(df)

354

In [10]:
df.columns

Index(['Sector', 'Project Name', 'Type', 'URL', 'Date', 'File Size',
       'Document_ID', 'Recommendation', 'Description of Recommendation',
       'Feasibility_Study_Complete', 'EIIR_Estimated', 'EIIR_Above_Viable',
       'Climate Change Mentioned'],
      dtype='object')

In [12]:
results_df.rename(columns = {'Project_Name': 'Project Name'}, inplace = True)

In [16]:
# Clean the merged dataset by combining duplicate columns
def clean_merged_data(df):
    """
    Clean the merged dataset by combining duplicate columns
    Priority: _x columns take precedence over _y columns
    """
    
    # Create a copy to avoid modifying the original
    cleaned_df = df.copy()
    
    print("Starting data cleaning process...")
    print(f"Original shape: {df.shape}")
    
    # Function to merge columns with priority to _x
    def merge_columns(col_x, col_y, df, new_col_name):
        """
        Merge two columns, prioritizing _x column values
        """
        # Create the new column
        df[new_col_name] = df[col_x]
        
        # Fill NaN values in _x with values from _y
        mask = df[col_x].isna() & df[col_y].notna()
        df.loc[mask, new_col_name] = df.loc[mask, col_y]
        
        # Check for conflicts (both columns have non-null values that differ)
        conflicts = df[(df[col_x].notna()) & (df[col_y].notna()) & (df[col_x] != df[col_y])]
        
        if len(conflicts) > 0:
            print(f"\nConflicts found in {new_col_name}:")
            print(f"Number of conflicts: {len(conflicts)}")
            for idx, row in conflicts.head(5).iterrows():
                print(f"  Row {idx}: '{row[col_x]}' vs '{row[col_y]}'")
            if len(conflicts) > 5:
                print(f"  ... and {len(conflicts) - 5} more conflicts")
        
        return df
    
    # Merge the duplicate columns
    print("\nMerging duplicate columns...")
    
    # 1. Recommendation columns
    cleaned_df = merge_columns('Recommendation_x', 'Recommendation_y', cleaned_df, 'Recommendation')
    
    # 2. Feasibility Study Complete columns
    cleaned_df = merge_columns('Feasibility_Study_Complete_x', 'Feasibility_Study_Complete_y', cleaned_df, 'Feasibility_Study_Complete')
    
    # 3. EIIR Estimated columns
    cleaned_df = merge_columns('EIIR_Estimated_x', 'EIIR_Estimated_y', cleaned_df, 'EIIR_Estimated')
    
    # 4. EIIR Above Viable columns
    cleaned_df = merge_columns('EIIR_Above_Viable_x', 'EIIR_Above_Viable_y', cleaned_df, 'EIIR_Above_Viable')
    
    # 5. Climate Change columns
    cleaned_df = merge_columns('Climate Change Mentioned', 'Climate_Change_Risk_Mentioned', cleaned_df, 'Climate_Change_Mentioned')
    
    # Drop the original duplicate columns
    columns_to_drop = [
        'Recommendation_x', 'Recommendation_y',
        'Feasibility_Study_Complete_x', 'Feasibility_Study_Complete_y',
        'EIIR_Estimated_x', 'EIIR_Estimated_y',
        'EIIR_Above_Viable_x', 'EIIR_Above_Viable_y',
        'Climate Change Mentioned', 'Climate_Change_Risk_Mentioned'
    ]
    
    cleaned_df = cleaned_df.drop(columns=columns_to_drop)
    
    print(f"\nCleaned shape: {cleaned_df.shape}")
    print(f"Dropped {len(columns_to_drop)} duplicate columns")
    
    # Display final column structure
    print(f"\nFinal columns ({len(cleaned_df.columns)}):")
    for i, col in enumerate(cleaned_df.columns, 1):
        print(f"{i:2d}. {col}")
    
    return cleaned_df

# Clean the data
cleaned_final_df = clean_merged_data(final_df)


Starting data cleaning process...
Original shape: (354, 21)

Merging duplicate columns...

Cleaned shape: (354, 16)
Dropped 10 duplicate columns

Final columns (16):
 1. Sector
 2. Project Name
 3. Type
 4. URL
 5. Date
 6. File Size
 7. Document_ID
 8. Description of Recommendation
 9. Recommendation_Description
10. Climate_Change_Risk_Descriptions
11. llm_model
12. Recommendation
13. Feasibility_Study_Complete
14. EIIR_Estimated
15. EIIR_Above_Viable
16. Climate_Change_Mentioned


In [17]:
# Analyze data quality and save the cleaned dataset
def analyze_data_quality(df):
    """
    Analyze the quality of the cleaned data
    """
    print("\n" + "="*60)
    print("DATA QUALITY ANALYSIS")
    print("="*60)
    
    # Basic statistics
    print(f"Total records: {len(df)}")
    print(f"Total columns: {len(df.columns)}")
    
    # Check for missing values in key columns
    key_columns = ['Recommendation', 'Feasibility_Study_Complete', 'EIIR_Estimated', 
                   'EIIR_Above_Viable', 'Climate_Change_Mentioned']
    
    print(f"\nMissing values in key columns:")
    for col in key_columns:
        if col in df.columns:
            missing_count = df[col].isna().sum()
            missing_pct = (missing_count / len(df)) * 100
            print(f"  {col}: {missing_count} ({missing_pct:.1f}%)")
    
    # Check unique values in categorical columns
    categorical_columns = ['Recommendation', 'Feasibility_Study_Complete', 'Climate_Change_Mentioned']
    
    print(f"\nUnique values in categorical columns:")
    for col in categorical_columns:
        if col in df.columns:
            unique_vals = df[col].value_counts()
            print(f"\n  {col}:")
            for val, count in unique_vals.head(10).items():
                print(f"    {val}: {count}")
            if len(unique_vals) > 10:
                print(f"    ... and {len(unique_vals) - 10} more unique values")
    
    # Check sectors
    print(f"\nProjects by Sector:")
    sector_counts = df['Sector'].value_counts()
    for sector, count in sector_counts.items():
        print(f"  {sector}: {count}")
    
    return df

# Analyze the cleaned data
cleaned_final_df = analyze_data_quality(cleaned_final_df)

# Save the cleaned dataset
output_filename = 'MSFR_Data_Cleaned.xlsx'
cleaned_final_df.to_excel(output_filename, index=False)
print(f"\nCleaned dataset saved to: {output_filename}")

# Also save as CSV for easier access
csv_filename = 'MSFR_Data_Cleaned.csv'
cleaned_final_df.to_csv(csv_filename, index=False, encoding='utf-8')
print(f"Also saved as CSV: {csv_filename}")

# Display first few rows of cleaned data
print(f"\nFirst 5 rows of cleaned data:")
cleaned_final_df.head()



DATA QUALITY ANALYSIS
Total records: 354
Total columns: 16

Missing values in key columns:
  Recommendation: 288 (81.4%)
  Feasibility_Study_Complete: 288 (81.4%)
  EIIR_Estimated: 288 (81.4%)
  EIIR_Above_Viable: 288 (81.4%)
  Climate_Change_Mentioned: 288 (81.4%)

Unique values in categorical columns:

  Recommendation:
    Neutral: 42
    Positive: 20
    Negative: 4

  Feasibility_Study_Complete:
    No: 39
    Not specified: 17
    Yes: 10

  Climate_Change_Mentioned:
    No: 66

Projects by Sector:
  Doprava: 148
  Informatizacia: 112
  Budovy: 44
  Obrana: 28
  Ostatne: 22

Cleaned dataset saved to: MSFR_Data_Cleaned.xlsx
Also saved as CSV: MSFR_Data_Cleaned.csv

First 5 rows of cleaned data:


,Sector,Project Name,Type,URL,Date,File Size,Document_ID,Description of Recommendation,Recommendation_Description,Climate_Change_Risk_Descriptions,llm_model,Recommendation,Feasibility_Study_Complete,EIIR_Estimated,EIIR_Above_Viable,Climate_Change_Mentioned
0,Obrana,UNKNOWN_PROJECT,Dohody o hraničných priechodoch,https://www.mfsr.sk/sk/dane-cla-uctovnictvo/co...,NaN,NaN,4.023,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Obrana,Rekonštrukcia leteckej základne Sliač (aktuali...,hodnotenie,https://www.mfsr.sk/files/archiv/28/mo_sliac_a...,23.1.2025,140 kB,4.016,NaN,The assessment does not provide an explicit go...,No climate-related risks are discussed; the ri...,gpt-5,Neutral,Not specified,Not specified,Not specified - The assessment does not report...,No
2,Obrana,Rekonštrukcia leteckej základne Sliač (aktuali...,štúdia uskutočniteľnosti,https://www.mosr.sk/data/files/5761_priloha-7-...,NaN,NaN,4.025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Obrana,Rekonštrukcia leteckej základne Sliač (aktuali...,hodnotenie,https://www.mfsr.sk/files/archiv/28/MO_Sliac_a...,4.10.2024,165 kB,4.015,NaN,The assessment recognizes the project as a MO ...,The assessment does not mention climate change...,gpt-5,Neutral,Yes,Not specified,Not specified - The document provides no IRR/B...,No
4,Obrana,Rekonštrukcia leteckej základne Sliač (aktuali...,štúdia uskutočniteľnosti,https://www.mosr.sk/data/files/5691_semai-195-...,NaN,NaN,4.027,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
results_df['llm_model'] = 'gpt-5'

In [13]:
final_df = pd.merge(df, results_df,  on = ['Project Name', 'Document_ID', 'URL'], how='left')

In [15]:
final_df.columns

Index(['Sector', 'Project Name', 'Type', 'URL', 'Date', 'File Size',
       'Document_ID', 'Recommendation_x', 'Description of Recommendation',
       'Feasibility_Study_Complete_x', 'EIIR_Estimated_x',
       'EIIR_Above_Viable_x', 'Climate Change Mentioned', 'Recommendation_y',
       'Recommendation_Description', 'Feasibility_Study_Complete_y',
       'EIIR_Estimated_y', 'EIIR_Above_Viable_y',
       'Climate_Change_Risk_Mentioned', 'Climate_Change_Risk_Descriptions',
       'llm_model'],
      dtype='object')

In [33]:
def analyze_document_by_id(document_id, custom_prompt=None, model="gpt-4.1"):
    """
    Analyze a specific document by Document ID with a custom prompt
    
    Parameters:
    - document_id: The Document ID to analyze (e.g., "4.003", "3.015")
    - custom_prompt: Custom prompt to use. If None, uses default analysis prompt
    - model: OpenAI model to use (default: "gpt-4o")
    
    Returns:
    - Dictionary with analysis results
    """
    
    try:
        # Construct the path to the PDF file
        pdf_path = f"downloaded_first_links/{document_id}.pdf"
        
        # Check if file exists
        if not os.path.exists(pdf_path):
            return {
                "error": f"PDF file not found: {pdf_path}",
                "document_id": document_id,
                "status": "failed"
            }
        
        # Extract text from PDF
        pdf_text = extract_text_from_pdf(pdf_path)
        
        if pdf_text.startswith("Error reading PDF"):
            return {
                "error": pdf_text,
                "document_id": document_id,
                "status": "failed"
            }
    
        
        # Use custom prompt or default prompt
        if custom_prompt is None:
            prompt = f"""This document is an assessment of a Cost Benefit Analysis project in Slovak language. 
            Read through the document and extract the following information - Extract only in JSON file. 
            Make sure that the responses come from the document and do not make anything up.

            Document ID: {document_id}

            Document Text:
            {pdf_text}

            Please extract:
            - Positive/Negative/Neutral Recommendation of the Project - Provide some additional information about the recommendation to support the answer
            - Is the Feasibility Study Considered Complete?
            - EIIR estimated or CBA Estimate Result - Estimated EIIR internal rate of return value based on the document/or Benefit-Cost Ratio and project being analyzed
            - EIIR above deemed viable - Provide some additional information about the recommendation to support the answer
            - Climate Change Risk Mentioned - Yes/No on whether the assessment mentions climate change risk
            - Climate Change Risk Mentioned Descriptions - Provide some additional information about the climate change risk to support the answer

            All responses should be in English. Return the response in JSON format with these exact keys:
            {{
                "recommendation": "Positive/Negative/Neutral"
                "recommendation_description": "Additional information about the recommendation to support the answer",
                "feasibility_study_complete": "Yes/No/Not specified",
                "eiir_estimated": "value or Not specified",
                "eiir_above_viable": "Yes/No/Not specified - provide some additional information about the recommendation to support the answer",
                "climate_change_risk_mentioned": "Yes/No",
                "climate_change_risk_mentioned_descriptions": "Additional information about the climate change risk to support the answer"
            }}"""
        else:
            prompt = f"""Document ID: {document_id}

            Document Text:
            {pdf_text}

            {custom_prompt}"""
        
        # Call OpenAI API
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are an expert at analyzing Slovak government documents and extracting structured information."},
                {"role": "user", "content": prompt}
            ]
        )
        
        return {
            "document_id": document_id,
            "pdf_path": pdf_path,
            "analysis": response.choices[0].message.content,
            "model_used": model,
            "text_length": len(pdf_text),
            "status": "success"
        }
        
    except Exception as e:
        return {
            "error": f"Error analyzing document: {str(e)}",
            "document_id": document_id,
            "status": "failed"
        }



In [24]:
# Example usage and helper functions

def get_project_info(document_id):
    """
    Get project information for a given document ID
    """
    project_info = filtered_df[filtered_df['First_File_Document_ID'] == document_id]
    if len(project_info) > 0:
        return project_info.iloc[0]
    else:
        return None

def analyze_with_custom_prompt(document_id, prompt_text, model="gpt-4.1"):
    """
    Convenience function to analyze a document with a custom prompt
    """
    result = analyze_document_by_id(document_id, custom_prompt=prompt_text, model=model)
    
    if result['status'] == 'success':
        print(f"Analysis for Document ID: {document_id}")
        print("=" * 50)
        
        # Get project info
        project_info = get_project_info(document_id)
        if project_info is not None:
            print(f"Project Name: {project_info['Project_Name']}")
            print(f"URL: {project_info['First_File_URL']}")
        
        print(f"Model Used: {result['model_used']}")
        print(f"Text Length: {result['text_length']} characters")
        print("\nAnalysis Result:")
        print("-" * 30)
        print(result['analysis'])
        
        return result
    else:
        print(f"Error analyzing document {document_id}: {result['error']}")
        return result

# Example usage examples (commented out - uncomment to run)
"""
# Example 1: Analyze with default prompt
result = analyze_document_by_id('4.003')
print(result['analysis'])

# Example 2: Analyze with custom prompt
result = analyze_with_custom_prompt('4.003', 'Summarize the main findings in 3 bullet points')

# Example 3: Analyze with different model
result = analyze_document_by_id('3.015', model='gpt-4o', max_chars=12000)

# Example 4: Get specific information
result = analyze_with_custom_prompt('4.003', 'What is the total project cost mentioned in this document?')

# Example 5: Compare projects
result1 = analyze_with_custom_prompt('4.003', 'What are the main risks identified in this project?')
result2 = analyze_with_custom_prompt('3.015', 'What are the main risks identified in this project?')
"""

print("Function ready! Use analyze_document_by_id() or analyze_with_custom_prompt() to analyze documents.")
print("Available document IDs shown above.")


Function ready! Use analyze_document_by_id() or analyze_with_custom_prompt() to analyze documents.
Available document IDs shown above.


In [39]:
prompt = f'''What are the key recommendations made by the MoF for this project?'''
doc_id = '3.012'


response = analyze_with_custom_prompt(doc_id, prompt)
response['analysis']

Analysis for Document ID: 3.012
Model Used: gpt-4.1
Text Length: 33110 characters

Analysis Result:
------------------------------
**Key Recommendations of the Ministry of Finance (MF SR) for the project "Reforma súdnej mapy v Bratislave":**

1. **Reduce overall project costs by at least 8.6 million EUR (without VAT)**  
   - Achieve this by negotiating a reduction of the purchase price by the estimated fit-out/adaptation costs.

2. **Reconsider the selection of the specific building**  
   - Choosing a different building than the currently preferred one could yield additional savings of 8.7 million EUR (without VAT).

3. **Adjust the scope of the project to achieve theoretical savings of 5.5 million EUR**  
   - Alternatively, seek options for effective use of surplus space above the Ministry of Justice’s requirements (almost 3,000 m²), e.g., through commercial rental (such as for a cafeteria/restaurant) or by placing another state institution in the building.

4. **Obtain an independ

'**Key Recommendations of the Ministry of Finance (MF SR) for the project "Reforma súdnej mapy v Bratislave":**\n\n1. **Reduce overall project costs by at least 8.6 million EUR (without VAT)**  \n   - Achieve this by negotiating a reduction of the purchase price by the estimated fit-out/adaptation costs.\n\n2. **Reconsider the selection of the specific building**  \n   - Choosing a different building than the currently preferred one could yield additional savings of 8.7 million EUR (without VAT).\n\n3. **Adjust the scope of the project to achieve theoretical savings of 5.5 million EUR**  \n   - Alternatively, seek options for effective use of surplus space above the Ministry of Justice’s requirements (almost 3,000 m²), e.g., through commercial rental (such as for a cafeteria/restaurant) or by placing another state institution in the building.\n\n4. **Obtain an independent property valuation (at least one RICS-standard appraisal)**  \n   - This should be done before negotiations with th

Analysis Result:
------------------------------
**Key Recommendations of the Ministry of Finance (MF SR) for the project "Reforma súdnej mapy v Bratislave":**

1. **Reduce overall project costs by at least 8.6 million EUR (without VAT)**  
   - Achieve this by negotiating a reduction of the purchase price by the estimated fit-out/adaptation costs.

2. **Reconsider the selection of the specific building**  
   - Choosing a different building than the currently preferred one could yield additional savings of 8.7 million EUR (without VAT).

3. **Adjust the scope of the project to achieve theoretical savings of 5.5 million EUR**  
   - Alternatively, seek options for effective use of surplus space above the Ministry of Justice’s requirements (almost 3,000 m²), e.g., through commercial rental (such as for a cafeteria/restaurant) or by placing another state institution in the building.

4. **Obtain an independent property valuation (at least one RICS-standard appraisal)**  
   - This should be done before negotiations with the property owner on price reduction.

5. **Commercialize surplus parking spaces**  
   - Any parking beyond the needs of court employees (potentially up to 150 spaces) should be rented out, sold, or made accessible to the public.

6. **Offer surplus court buildings to other state institutions or sell them commercially if unneeded**  
   - The estimated potential value is about 28 million EUR (without VAT). A more precise estimate requires expert property appraisals.

7. **After preparing project documentation, submit reconstruction projects for Bratislava I & II City Courts and the Bratislava Administrative Court for MoF’s economic review**.

8. **Continue project preparation and submit an updated feasibility study for MoF assessment**  
   - This should be done before a binding commitment is made under §19a par. 4 of law no. 523/2004 Coll.

---

**In summary:** The MF SR recommends reducing costs via negotiation and building choice, maximizing the use of surplus space and parking, verifying property values independently, and following proper procedures for reconstruction and asset disposition. All steps should be taken with close regard to deadlines and the requirements of the Recovery and Resilience Plan.

In [43]:
prompt = f''' To what extent were these recommendations followed:
Key recommendations made by the Ministry of Finance (MoF) – ÚHP for this project:

Before signing the contract for the purchase of the building:
1. Verify the availability of funds from the Recovery and Resilience Plan (POO) – and ensure these funds cover, in addition to the purchase, any other necessary expenditures needed to put the Bratislava III and IV city courts into operation (e.g., necessary construction modifications, interior equipment, etc.).
2. Inform the Ministry of Finance about the plan for using the vacated buildings. Redundant buildings should first be offered to other state institutions, and if there is no interest, they should be sold commercially.
3. Submit the updated feasibility study for evaluation by the Ministry of Finance under §19a section 4b of Act No. 523/2004 Coll. on budgetary rules of public administration (and subsequent amendments).

Summary:  
- Ensure all costs (not just purchase price) are covered by available funding.  
- Provide information and seek optimal use or sale of vacated court buildings.  
- Submit updated project documentation for formal MoF assessment and compliance with legal requirements.'''
doc_id = '3.032'

response = analyze_with_custom_prompt(doc_id, prompt)
response['analysis']

Analysis for Document ID: 3.032
Model Used: gpt-4.1
Text Length: 6666 characters

Analysis Result:
------------------------------
Here is a structured analysis of compliance with the Ministry of Finance (ÚHP) key recommendations for Document ID: 3.032 (Reforma súdnej mapy v Bratislave):

---

### 1. **Verification of Availability of Funding for All Project Needs (purchase & operationalization)**
**Recommendation:**  
Verify availability of Recovery and Resilience Plan funds (POO), ensuring they cover the purchase plus all other expenditures required to put the courts into operation (construction, equipment, etc.).

**Extent of Implementation:**  
**Partially followed.**

**Evidence from Document:**
- The document details the sources:  
   - POO funds allocated for the purchase (€46.2m),  
   - Additional POO funds for covering a €1.6m shortfall by redirecting from unspent court reconstruction allocations,  
   - DPH (€9.5m) paid from the state budget.
- The document explicitly mentions

'Here is a structured analysis of compliance with the Ministry of Finance (ÚHP) key recommendations for Document ID: 3.032 (Reforma súdnej mapy v Bratislave):\n\n---\n\n### 1. **Verification of Availability of Funding for All Project Needs (purchase & operationalization)**\n**Recommendation:**  \nVerify availability of Recovery and Resilience Plan funds (POO), ensuring they cover the purchase plus all other expenditures required to put the courts into operation (construction, equipment, etc.).\n\n**Extent of Implementation:**  \n**Partially followed.**\n\n**Evidence from Document:**\n- The document details the sources:  \n   - POO funds allocated for the purchase (€46.2m),  \n   - Additional POO funds for covering a €1.6m shortfall by redirecting from unspent court reconstruction allocations,  \n   - DPH (€9.5m) paid from the state budget.\n- The document explicitly mentions €0.9m from POO earmarked for additional technical equipment (notably a new archive shelving system), and states 

In [18]:
import pandas as pd

df_final = pd.read_csv('MSFR_Data_AllTypes - MSFR_Data_Cleaned (1).csv')
df_new = pd.read_csv('link_text_and_urls_PIIAG Slovakia Project Analysis 2025-09-13 (002).docx.txt')

In [7]:
df_final.columns

Index(['Sector', 'Project Name', 'Analysis', 'Analysis Information Extracted',
       'Type', 'Type English', 'URL', 'Date', 'File Size', 'Document_ID',
       'Project_ID', 'Feasibility Study Type', 'Recommendation',
       'Description of Recommendation', 'Feasibility_Study_Complete',
       'EIIR_Estimated', 'EIIR_Above_Viable', 'Climate Change Mentioned',
       'LLM Model Used', 'Unnamed: 19'],
      dtype='object')

In [19]:
df_new.columns

Index(['Assessment Online Disclosure | Assessment Online Disclosure | https://www.mfsr.sk/uhp/hodnotenia/'], dtype='object')

In [20]:
df_new['Link Text'].unique()

KeyError: 'Link Text'